# CELL 1 :Imports for NumPy and Matplotlib for all plots 

In [2]:
import csv
import time
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
rng = np.random.default_rng(42)

CSV_PATH = "Education in General.csv"
print("NumPy", np.__version__)

NumPy 2.3.3


# CELL 2 : Load the CSV and drop the two empty trailing columns. utf-8-sig removes the BOM.

In [3]:
MISSING_TOKENS = {"#N/B", "", "NA", "N/A", "nan", "NaN", "..", "-"}

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))

header, data_rows = rows[0], rows[1:]
print("Raw shape: %d rows x %d columns" % (len(data_rows), len(header)))

keep = [j for j, name in enumerate(header) if name.strip() and not name.startswith("Unnamed")]
header    = [header[j] for j in keep]
data_rows = [[r[j] if j < len(r) else "" for j in keep] for r in data_rows]
n_rows = len(data_rows)
print("After dropping empty columns: %d columns" % len(header))
for h in header: print("  -", h)

Raw shape: 756 rows x 13 columns
After dropping empty columns: 11 columns
  - ISO_Code
  - Country
  - Year
  - School life expectancy, primary to tertiary, male (years)
  - School life expectancy, primary to tertiary, female (years)
  - Government expenditure on primary education, US$ (millions)
  - Government expenditure on secondary education, US$ (millions)
  - Government expenditure on tertiary education, US$ (millions)
  - Government expenditure on primary education as a percentage of GDP (%)
  - Government expenditure on secondary education as a percentage of GDP (%)
  - Government expenditure on tertiary education as a percentage of GDP (%)


# CELL 3 : Detect non-numeric columns automatically (ISO_Code, Country come out as categorical).

In [4]:
def is_number(s):
    s = s.strip()
    if s in MISSING_TOKENS:
        return True
    try:
        float(s); return True
    except ValueError:
        return False

col_numeric = []
for j in range(len(header)):
    col_numeric.append(all(is_number(data_rows[i][j]) for i in range(n_rows)))

numeric_cols     = [header[j] for j in range(len(header)) if col_numeric[j]]
categorical_cols = [header[j] for j in range(len(header)) if not col_numeric[j]]
print("NON-NUMERIC (categorical) columns :", categorical_cols)
print("NUMERIC columns                   :", len(numeric_cols), "found")

NON-NUMERIC (categorical) columns : ['ISO_Code', 'Country']
NUMERIC columns                   : 9 found


# CELL 4 : Encode the categorical columns with label encoding. Country is an identifier, so we encode it but keep it only as a label, not as a PCA feature.

In [5]:
col = {name: j for j, name in enumerate(header)}

def label_encode(values):
    cats = sorted(set(values))
    mapping = {c: i for i, c in enumerate(cats)}
    return np.array([mapping[v] for v in values]), mapping

countries = [data_rows[i][col["Country"]] for i in range(n_rows)]
country_codes, country_map = label_encode(countries)
years = np.array([int(float(data_rows[i][col["Year"]])) for i in range(n_rows)])

period = np.select([years <= 2014, years <= 2019], [0, 1], default=2)
period_names = ["2010–2014", "2015–2019", "2020–2023"]

print("Label-encoded %d countries (codes 0..%d). Sample:" % (len(country_map), len(country_map)-1))
for c in list(country_map)[:5]:
    print("   %-15s -> %d" % (c, country_map[c]))

Label-encoded 54 countries (codes 0..53). Sample:
   Algeria         -> 0
   Angola          -> 1
   Benin           -> 2
   Botswana        -> 3
   Burkina Faso    -> 4


# CELL 5 : Build the 8-feature numeric matrix X and turn '#N/B' markers into NaN. Year is excluded.

In [6]:
feature_cols = [c for c in numeric_cols if c != "Year"]
feat_idx = [col[c] for c in feature_cols]

def to_float(s):
    s = s.strip()
    return np.nan if s in MISSING_TOKENS else float(s)

X_raw = np.array([[to_float(data_rows[i][j]) for j in feat_idx] for i in range(n_rows)], float)

print("Feature matrix X: %d rows x %d features\n" % X_raw.shape)
print("Missing values per feature (was '#N/B'):")
for name, m in zip(feature_cols, np.isnan(X_raw).sum(0)):
    print("   %3d  (%.0f%%)  %s" % (m, 100*m/n_rows, name))

Feature matrix X: 756 rows x 8 features

Missing values per feature (was '#N/B'):
   496  (66%)  School life expectancy, primary to tertiary, male (years)
   496  (66%)  School life expectancy, primary to tertiary, female (years)
   504  (67%)  Government expenditure on primary education, US$ (millions)
   500  (66%)  Government expenditure on secondary education, US$ (millions)
   509  (67%)  Government expenditure on tertiary education, US$ (millions)
   450  (60%)  Government expenditure on primary education as a percentage of GDP (%)
   443  (59%)  Government expenditure on secondary education as a percentage of GDP (%)
   448  (59%)  Government expenditure on tertiary education as a percentage of GDP (%)


# CELL 6 : Impute missing values with the column MEDIAN (robust to the outlier % -of-GDP errors).

In [7]:
def median_impute(X):
    X = X.copy()
    med = np.nanmedian(X, axis=0)
    nan_r, nan_c = np.where(np.isnan(X))
    X[nan_r, nan_c] = med[nan_c]
    return X, med

X_imp, medians = median_impute(X_raw)
print("NaNs before:", int(np.isnan(X_raw).sum()), "| after imputation:", int(np.isnan(X_imp).sum()))

NaNs before: 3846 | after imputation: 0


# CELL 7 : Standardize to mean 0, std 1. Needed because features have different units  (years vs US$ millions vs %); otherwise big-number columns would dominate PCA.

In [8]:
def standardize(X):
    mu = X.mean(0)
    sd = X.std(0, ddof=0); sd[sd == 0] = 1.0
    return (X - mu) / sd, mu, sd

X_std, mu, sd = standardize(X_imp)
print("Per-feature mean after scaling (≈0):", X_std.mean(0))
print("Per-feature std  after scaling (≈1):", X_std.std(0))

Per-feature mean after scaling (≈0): [ 0. -0.  0. -0. -0. -0. -0. -0.]
Per-feature std  after scaling (≈1): [1. 1. 1. 1. 1. 1. 1. 1.]


# CELL 8 : TASK 1: PCA from scratch. Center -> covariance matrix -> eigen-decomposition (eigh) -> sort eigenvalues descending -> project. Eigenvectors = principal components, eigenvalues = variance along each.

In [9]:
def pca_fit(X, n_components=None):
    '''PCA from scratch. Returns (scores, eigenvalues, eigenvectors, covariance).'''
    n = X.shape[0]
    Xc  = X - X.mean(0)                       # 1. center
    cov = (Xc.T @ Xc) / (n - 1)               # 2. covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov)    # 3. eigh for symmetric matrices
    order   = np.argsort(eigvals)[::-1]       # 4. sort DESCENDING
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    if n_components is not None:
        eigvals, eigvecs = eigvals[:n_components], eigvecs[:, :n_components]
    scores = Xc @ eigvecs                     # 5. project
    return scores, eigvals, eigvecs, cov

scores, eigvals, eigvecs, cov = pca_fit(X_std)

print("Covariance matrix shape:", cov.shape)
print("\nEigenvalues (variance per component, descending):")
print(eigvals)
print("\nSorted descending? ->", bool(np.all(np.diff(eigvals) <= 1e-9)))

Covariance matrix shape: (8, 8)

Eigenvalues (variance per component, descending):
[3.0634 3.0008 1.7339 0.1371 0.0653 0.0101 0.0001 0.    ]

Sorted descending? -> True


# CELL 9 :TASK 2: explained variance per component + cumulative. We keep the SMALLEST number of components reaching 95% variance -> k = 3. TRADEOFF: 3 components keep ~97% of the information in a simpler, de-noised form, but we drop the last ~3% of variance and lose the literal meaning of each feature (each PC is now a mix of all 8). We chose 95% to balance simplicity vs faithfulness.

In [ ]:
evr = eigvals / eigvals.sum()
cum = np.cumsum(evr)

print(" PC   eigenvalue   explained%   cumulative%")
for i, (l, e, c) in enumerate(zip(eigvals, evr, cum), 1):
    print("  %d   %9.4f   %8.2f%%   %9.2f%%" % (i, l, 100*e, 100*c))

def select_k(eigvals, threshold=0.95):
    '''Smallest k whose cumulative explained variance reaches `threshold`.'''
    cum = np.cumsum(eigvals / eigvals.sum())
    return int(np.searchsorted(cum, threshold) + 1)

THRESHOLD = 0.95
k = select_k(eigvals, THRESHOLD)
print("\nComponents to retain %.0f%% variance: k = %d (out of %d features)" % (100*THRESHOLD, k, len(eigvals)))
print("Variance actually retained at k=%d: %.2f%%" % (k, 100*cum[k-1]))

 PC   eigenvalue   explained%   cumulative%
  1      3.0634      38.24%       38.24%
  2      3.0008      37.46%       75.70%
  3      1.7339      21.64%       97.35%
  4      0.1371       1.71%       99.06%
  5      0.0653       0.81%       99.87%
  6      0.0101       0.13%      100.00%
  7      0.0001       0.00%      100.00%
  8      0.0000       0.00%      100.00%

Components to retain 95% variance: k = 3 (out of 8 features)
Variance actually retained at k=3: 97.35%

# CELL 10 : Scree + cumulative variance plots. The "elbow" after PC3 and the 95% line justify k=3.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
xs = np.arange(1, len(evr) + 1)

ax[0].bar(xs, 100*evr, color="#4C72B0")
ax[0].plot(xs, 100*evr, "o-", color="#C44E52")
ax[0].set(title="Scree plot — variance per component",
          xlabel="Principal component", ylabel="Explained variance (%)")
ax[0].set_xticks(xs)

ax[1].plot(xs, 100*cum, "o-", color="#55A868")
ax[1].axhline(100*THRESHOLD, ls="--", color="grey", label="%.0f%% threshold" % (100*THRESHOLD))
ax[1].axvline(k, ls=":", color="#C44E52", label="chosen k = %d" % k)
ax[1].set(title="Cumulative explained variance",
          xlabel="Number of components", ylabel="Cumulative variance (%)")
ax[1].set_xticks(xs); ax[1].legend()
plt.tight_layout(); plt.show()

# CELL 11 — Reconstruction error: rebuild the data from only k components. This IS the information lost = the ~2.65% of variance we discarded.

In [ ]:
scores_k, _, vecs_k, _ = pca_fit(X_std, n_components=k)
X_recon = scores_k @ vecs_k.T
mse = np.mean((X_std - X_recon) ** 2)
print("Reconstruction MSE keeping k=%d of %d components: %.4f" % (k, len(eigvals), mse))
print("Variance discarded (information lost): %.2f%%" % (100*(1 - cum[k-1])))

Reconstruction MSE keeping k=3 of 8 components: 0.0265
Variance discarded (information lost): 2.65%

# CELL 12 — TASK 3 VISUALIZATION (before vs after PCA). Left: two correlated original features -> tilted cloud (shows the redundancy).Right: same 756 points in PC1-PC2 space -> the cloud is just ROTATED and CENTERED, with PC1 holding the most variance. Same points, structure preserved, no data lost.

In [ ]:
fa = feature_cols.index("Government expenditure on primary education, US$ (millions)")
fb = feature_cols.index("Government expenditure on secondary education, US$ (millions)")
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
for g in range(3):
    m = period == g
    ax[0].scatter(X_std[m, fa], X_std[m, fb], s=18, alpha=0.55, color=colors[g], label=period_names[g])
    ax[1].scatter(scores[m, 0], scores[m, 1], s=18, alpha=0.55, color=colors[g], label=period_names[g])

ax[0].set(title="BEFORE PCA — original feature space",
          xlabel="Primary spending, US$m (standardized)",
          ylabel="Secondary spending, US$m (standardized)")
ax[1].set(title="AFTER PCA — principal-component space",
          xlabel="PC1 (%.1f%% variance)" % (100*evr[0]),
          ylabel="PC2 (%.1f%% variance)" % (100*evr[1]))
ax[1].axhline(0, color="grey", lw=.6); ax[1].axvline(0, color="grey", lw=.6)
for a in ax: a.legend(title="Year period", fontsize=8)
plt.tight_layout(); plt.show()

print("Same number of points in both plots: %d == %d" % (len(X_std), len(scores)))
print("PC1 variance (%.1f%%) > PC2 variance (%.1f%%): %s" % (100*evr[0], 100*evr[1], evr[0] > evr[1]))
print("PCA scores are centred at origin: mean PC1=%.2e, mean PC2=%.2e" % (scores[:,0].mean(), scores[:,1].mean()))

# CELL 13 — Loadings plot: shows what each PC means. PC1 ~ overall fiscal scale of education (the US$ columns); PC2 ~ spending intensity (% of GDP) vs attainment.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
y = np.arange(len(feature_cols))
ax.barh(y - 0.2, eigvecs[:, 0], height=0.4, color="#4C72B0", label="PC1")
ax.barh(y + 0.2, eigvecs[:, 1], height=0.4, color="#DD8452", label="PC2")
short = [c.replace("Government expenditure on ", "").replace(" education", "")
          .replace("School life expectancy, primary to tertiary, ", "SLE ") for c in feature_cols]
ax.set_yticks(y); ax.set_yticklabels(short, fontsize=8)
ax.axvline(0, color="grey", lw=.6)
ax.set(title="Component loadings — what PC1 and PC2 represent", xlabel="Loading (eigenvector weight)")
ax.legend(); plt.tight_layout(); plt.show()

# CELL 14 — Performance: equivalent PCA via SVD + a batched covariance that streams the data in chunks (handles datasets too big for memory). Prints confirm all routes agree.

In [ ]:
def pca_svd(X, n_components=None):
    '''Equivalent PCA via SVD of the centred data (no explicit covariance).'''
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    eigvals = (S ** 2) / (X.shape[0] - 1)
    V = Vt.T
    if n_components is not None:
        eigvals, V = eigvals[:n_components], V[:, :n_components]
    return Xc @ V, eigvals, V

def batched_covariance(X, batch=10_000):
    '''Memory-light covariance via streaming sums — handles data larger than RAM.'''
    n, d = X.shape
    s, ss = np.zeros(d), np.zeros((d, d))
    for i in range(0, n, batch):
        b = X[i:i+batch]
        s  += b.sum(0)
        ss += b.T @ b
    mean = s / n
    return (ss - n * np.outer(mean, mean)) / (n - 1)

_, e_svd, _ = pca_svd(X_std)
cov_batch   = batched_covariance(X_std, batch=128)
print("eigh vs svd  | max eigenvalue diff : %.2e" % np.max(np.abs(eigvals - e_svd)))
print("full vs batched covariance | max diff: %.2e" % np.max(np.abs(cov - cov_batch)))

eigh vs svd  | max eigenvalue diff : 2.28e-15
full vs batched covariance | max diff: 3.77e-15

# CELL 15 :Benchmark covariance+eigh vs full SVD on growing datasets. cov+eigh wins when rows >> features.

In [ ]:
sizes, d = [2_000, 10_000, 50_000, 100_000], 50
t_cov, t_svd = [], []
for n in sizes:
    Xb = rng.standard_normal((n, d))
    t0 = time.perf_counter(); pca_fit(Xb); t_cov.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); pca_svd(Xb); t_svd.append(time.perf_counter() - t0)

print("  n rows     cov+eigh     SVD       speed-up")
for n, a, b in zip(sizes, t_cov, t_svd):
    print("  %-8d  %7.4fs   %7.4fs   %4.1fx" % (n, a, b, b / a))

  n rows     cov+eigh     SVD       speed-up
  2000       0.0057s    0.0282s    4.9x
  10000      0.0112s    0.0553s    4.9x
  50000      0.0395s    0.2849s    7.2x
  100000     0.0892s    0.6136s    6.9x

# CELL 16 :Plot the benchmark timings.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(sizes, t_cov, "o-", label="covariance + eigh", color="#4C72B0")
ax.plot(sizes, t_svd, "s-", label="full SVD",          color="#C44E52")
ax.set(title="PCA runtime vs dataset size (d = 50 features)",
       xlabel="number of rows", ylabel="time (seconds)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()